# Agents and Tool Use Fundamentals

## 1. What Is an AI Agent?
An AI agent is a goal-directed system that decides, acts, observes outcomes, and iterates until completion or stopping criteria. Unlike a single model call, an agent architecture includes control logic, tool interfaces, memory, and policy constraints. In interviews, distinguish capability from architecture: the base LLM provides reasoning and language generation, while the agent loop provides multi-step execution and coordination.

## 2. Why Agents Instead of Single Prompt Chains
Agents are useful when tasks require dynamic branching, external tools, or uncertain intermediate states. Linear chains are easier to debug but can be brittle when task flow changes at runtime. A strong interview response explains that agents trade predictability for flexibility; therefore, production design must compensate with guardrails, observability, and clear termination logic.

## 3. ReAct Pattern: Reason, Act, Observe
ReAct combines short reasoning with concrete tool actions and observation-driven updates. This pattern improves multi-step reliability because the model does not need to hallucinate intermediate facts; it can fetch evidence and recalculate. Interview tip: emphasize bounded loops and validation, otherwise ReAct systems can overthink, call wrong tools repeatedly, or fail silently.

In [ ]:
import math
import numpy as np

def calculator(expression: str) -> str:
    safe_scope = {'sqrt': math.sqrt}
    try:
        value = eval(expression, {'__builtins__': {}}, safe_scope)
        return f'calculator_result={value}'
    except Exception as e:
        return f'calculator_error={e}'

def search_tool(query: str) -> str:
    knowledge = {
        'react': 'ReAct alternates reasoning and tool usage.',
        'rag': 'RAG retrieves external knowledge before generation.',
        'memory': 'Agent memory can be short-term, episodic, or long-term.'
    }
    return knowledge.get(query.lower().strip(), 'no_result_found')

print(calculator('(15+5)*2'))
print(search_tool('memory'))

## 4. Tool Calling Contracts and Schemas
Tool calling works best when each tool has clear intent, typed arguments, strict validation, and bounded side effects. Ambiguous tool schemas cause model confusion and unstable behavior. In interviews, describe tool contracts as API design for model orchestration: narrow interfaces, deterministic return format, and explicit error channels improve reliability significantly.

## 5. Planning Loops and Task Decomposition
Agents benefit from explicit planning loops where broad goals are decomposed into verifiable subtasks. A planner can produce steps, and an executor can complete each step with tools and checks. Interviewers often ask about over-planning risk; mention that shallow plans with frequent re-evaluation are typically more robust than deep speculative plans in noisy environments.

In [ ]:
def choose_tool(user_query: str):
    q = user_query.lower()
    if any(k in q for k in ['calculate', 'sum', 'multiply', 'sqrt', ':']):
        return 'calculator'
    if any(k in q for k in ['search', 'find', 'lookup', 'what is']):
        return 'search'
    return 'none'

def rule_based_agent(user_query: str):
    tool = choose_tool(user_query)
    if tool == 'calculator':
        expr = user_query.split(':', 1)[-1].strip()
        obs = calculator(expr)
        return f'Thought: exact arithmetic needed\nAction: calculator\nObservation: {obs}'
    if tool == 'search':
        query = user_query.split(':', 1)[-1].strip()
        obs = search_tool(query)
        return f'Thought: factual lookup required\nAction: search_tool\nObservation: {obs}'
    return 'Thought: answer directly without tools.'

print(rule_based_agent('calculate: (8+2)*6'))
print('-' * 60)
print(rule_based_agent('search: ReAct'))

## 6. Agent vs Chain vs RAG
A chain is deterministic and best for fixed workflows; RAG is ideal for knowledge grounding with retrieval; agents are suited for dynamic multi-step action. In interviews, compare them by flexibility, observability burden, and safety surface area. The best answer is rarely "always use agents"; instead, explain architecture selection based on task complexity and operational requirements.

In [ ]:
def architecture_choice(needs_external_knowledge, needs_dynamic_actions, low_variance_workflow):
    if low_variance_workflow and not needs_dynamic_actions:
        return 'chain'
    if needs_external_knowledge and not needs_dynamic_actions:
        return 'rag'
    if needs_dynamic_actions:
        return 'agent'
    return 'chain'

examples = [
    (False, False, True),
    (True, False, False),
    (True, True, False),
]
for ex in examples:
    print(ex, '->', architecture_choice(*ex))

## 7. Memory Types and Practical Limits
Agent memory includes short-term context windows, episodic logs of recent interactions, and long-term stores of preferences or facts. Memory improves continuity but can inject stale or incorrect information if not curated. Interviewers appreciate explicit mention of memory lifecycle controls: TTL policies, correction workflows, user consent, and deletion support.

In [ ]:
class MemoryStore:
    def __init__(self, max_items=5):
        self.max_items = max_items
        self.items = []

    def add(self, note):
        self.items.append(note)
        if len(self.items) > self.max_items:
            self.items = self.items[-self.max_items:]

    def recall(self):
        return list(self.items)

memory = MemoryStore(max_items=3)
for n in ['user likes concise answers', 'prefers Python examples', 'works in fintech', 'avoid API keys']:
    memory.add(n)

print('Current memory snapshot:')
print(memory.recall())

## 8. Failure Modes in Agent Systems
Typical failure modes include infinite loops, hallucinated tool outputs, invalid arguments, unsafe actions, and context overflow. Production systems should include loop caps, timeout policies, and confidence-aware fallback to human escalation. In interviews, show you understand that agent quality is process quality: even strong final answers can be unreliable if the intermediate path is unsafe.

## 9. Guardrails for Tool-Using Agents
Guardrails should enforce input validation, output policy checks, least-privilege tool permissions, and explicit deny rules for high-risk commands. Tool wrappers are ideal places to enforce business rules because they provide deterministic checkpoints. Interview-ready framing: let the model propose actions, but let policy code decide whether actions are allowed.

In [ ]:
steps = np.arange(1, 9)
success = 1 - np.exp(-0.42 * steps)
risk = np.exp(-0.35 * steps)

import matplotlib.pyplot as plt
plt.figure(figsize=(7.2, 4))
plt.plot(steps, success, marker='o', label='Task success probability')
plt.plot(steps, risk, marker='s', label='Residual risk')
plt.xlabel('Planning iterations')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.title('Toy trade-off: iteration vs reliability')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Observability and Tracing
Agent observability should log user input class, selected tools, arguments, tool outputs, retry counts, and stop reason. Traces are essential for debugging and post-incident review because final answers alone hide process failures. Interviewers often ask what to monitor first: start with task success, tool error rate, loop count distribution, and safety policy violations.

## 11. Evaluation Methodology
Evaluate agents with scenario-based test suites that include normal tasks, adversarial prompts, and tool failure injection. Process-level metrics matter as much as final correctness: wrong tool choices, unnecessary loops, and unsafe action attempts should be scored explicitly. A strong interview answer includes offline regression plus online shadow testing before full rollout.

## 12. Cost, Latency, and Reliability Trade-offs
Agents may increase token usage and latency due to iterative reasoning and multiple tool calls. Production tuning includes smaller planning models, cached retrieval, and early-stop criteria when confidence is sufficient. In interviews, explain that architecture success depends on business SLOs; the best agent is one that meets quality goals within cost and latency budgets.

In [ ]:
calls = np.array([1, 2, 3, 4, 5, 6])
latency = 180 + 130 * calls
quality = 0.58 + 0.35 * (1 - np.exp(-0.55 * calls))

fig, ax1 = plt.subplots(figsize=(7.3, 4))
ax1.plot(calls, latency, marker='o', color='tab:red')
ax1.set_xlabel('Tool calls per request')
ax1.set_ylabel('Latency (ms)', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:red')

ax2 = ax1.twinx()
ax2.plot(calls, quality, marker='s', color='tab:blue')
ax2.set_ylabel('Quality score', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')

plt.title('Quality-latency balance in agent loops (illustrative)')
fig.tight_layout()
plt.show()

## 13. Common Agent Risks
Agent risks include unauthorized actions, overconfidence, brittle tool integration, hidden prompt injection through retrieved text, and compliance violations in logs. Effective risk management combines technical controls with process controls such as approval workflows and periodic red teaming. In interviews, explicitly mention least privilege and human override mechanisms.

## 14. Interview Answer Pattern for Agent Design Questions
A reliable structure is: define objective, select architecture (chain/RAG/agent), design tools and memory, add guardrails, define metrics, and plan rollout. This demonstrates both technical depth and product judgment. Interviewers usually prefer pragmatic design trade-offs over maximal complexity.

## 15. Interview Q&A (9 Detailed Questions)
**Q1: What is an AI agent?** A1: A system that iteratively reasons, acts through tools, observes outcomes, and updates decisions toward a goal.

**Q2: What is the ReAct pattern?** A2: A loop of reasoning plus tool actions, where observations guide the next step.

**Q3: Agent vs chain vs RAG?** A3: Chain for fixed flows, RAG for grounded knowledge retrieval, agent for dynamic multi-step actions.

**Q4: Why does tool calling improve reliability?** A4: Deterministic tools handle calculations and lookup better than free-form generation.

**Q5: What memory types matter most?** A5: Short-term context, episodic interaction memory, and bounded long-term user preferences.

**Q6: What are key agent limitations?** A6: Looping, tool misuse, latency growth, and vulnerability to injection attacks.

**Q7: How do you reduce agent risk?** A7: Tool schemas, permission boundaries, loop limits, output filters, and escalation rules.

**Q8: What metrics prove agent quality?** A8: End-task success, tool accuracy, loop efficiency, policy violation rate, and user satisfaction.

**Q9: What answer impresses in interviews?** A9: One that pairs conceptual patterns with concrete controls and measurable outcomes.

## 16. Summary and Key Interview Takeaways
Agents extend LLM capability through structured planning, tool integration, and memory-aware execution, but they also increase safety and operations complexity. The best production approach balances autonomy with deterministic control points.

**Key Takeaways:**
- ReAct is powerful when loops are bounded and observable.
- Tool contracts and permission scopes are core to reliability.
- Agent, chain, and RAG choices depend on task dynamics and grounding needs.
- Memory helps continuity but requires freshness and privacy controls.
- Interview excellence comes from architecture trade-offs plus risk mitigation.